# Bank Variant Modes — Testing the Variations System (v0.11.0, issue #53)

v0.11.0 added explicit **variant modes** to bank generation. The
undertrained unconditional DiT contracts every noise draw to (nearly)
one latent (#52), so raw DDIM draws yield n near-identical clips. The
decoder, however, trained with NOISE_INJECT, tolerates a latent
neighbourhood — so we vary *around the canonical draw* $\bar{z}$ and
decode each variant.

Interface (`LSDModel.generate_sound_bank`):

| `bank_mode` | latent rule | `bank_variety` |
|---|---|---|
| `canonical` (default) | n independent DDIM draws | — |
| `jitter` | $\bar{z} + \alpha\,\sigma_z\,\varepsilon_i$ | noise amplitude $\alpha$ |
| `residual` | $\bar{z} + k\,(z_i - \bar{z})$ | amplified residual rel. std |
| `stopvar` | same seed, step count swept 0.24–0.98·steps | unused |

Pre-registered gate (#53): **USEFUL iff pairwise L1 ≥ 0.05 AND FAD ≤ 1200
AND RMS ≥ 0.35 AND centroid spread > 20 Hz**. Gate-passing arms:
`jitter_a0.5` (L1 0.182), `resid_r0.3` (0.112), `stopvar` (0.073).

This notebook: loads the trained checkpoints, generates banks in every
mode, measures the gate metrics, sweeps the variety dial, writes
audition WAVs, and cross-checks against `results/bank_variants.csv`.

## Setup

Loads the prior, graph decoder and DiT from `results/artifacts/`
(produced by `scripts/run_evaluation.py`). MPS is used when available.

In [1]:
import random
import statistics
from pathlib import Path

import soundfile as sf
import torch

from ald_sc.audio_codec import EnCodecEncoder
from ald_sc.build_prior import build_arrow_prior
from ald_sc.dit import MinimalDiT
from ald_sc.eval import spectral_centroid
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.inference import BANK_MODES, LSDModel
from ald_sc.schedule import CosineSchedule

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS = REPO / 'results' / 'artifacts'
OUT_DIR = REPO / 'notebooks' / 'results' / 'bank_variants'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (ARTIFACTS / 'dit.pt').exists(), (
    'results/artifacts/ missing — run scripts/run_evaluation.py first'
)

SEED, N_BANK, STEPS = 3407, 8, 50
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
random.seed(SEED)
torch.manual_seed(SEED)
print(f'device={device}')

device=mps


In [2]:
prior = build_arrow_prior(
    torch.load(ARTIFACTS / 'embeddings.pt', weights_only=False), q=8, k=4
).to(device)
decoder = GraphDecoder(
    latent_channels=128, out_channels=1, feature_dim=128,
    base_channels=32, prior=prior, upsample_strides=(2, 4, 5, 8),
).to(device).eval()
decoder.load_state_dict(
    torch.load(ARTIFACTS / 'graph_dec.pt', weights_only=False, map_location='cpu')
)
dit = MinimalDiT(
    latent_channels=128, latent_length=300, patch_size=8,
    dim=64, depth=2, num_heads=4, spec_dim=24,
).to(device).eval()
dit.load_state_dict(
    torch.load(ARTIFACTS / 'dit.pt', weights_only=False, map_location='cpu')
)
encoder = EnCodecEncoder().to(device).eval()
model = LSDModel(
    prior=prior, dit=dit, decoder=decoder, encoder=encoder,
    schedule=CosineSchedule(num_steps=1000),
)
print(f'model loaded — BANK_MODES = {BANK_MODES}')

model loaded — BANK_MODES = ('canonical', 'jitter', 'residual', 'stopvar')


## Generate a bank in every mode

Defaults follow the gate-passing arms: `jitter` at variety 0.5,
`residual` at 0.3. Each bank is n=8 clips, 50 DDIM steps.

In [3]:
banks = {
    'canonical': model.generate_sound_bank(n=N_BANK, steps=STEPS, seed=SEED),
    'jitter_0.5': model.generate_sound_bank(
        n=N_BANK, steps=STEPS, seed=SEED, bank_mode='jitter', bank_variety=0.5
    ),
    'residual_0.3': model.generate_sound_bank(
        n=N_BANK, steps=STEPS, seed=SEED, bank_mode='residual', bank_variety=0.3
    ),
    'stopvar': model.generate_sound_bank(
        n=N_BANK, steps=STEPS, seed=SEED, bank_mode='stopvar'
    ),
}
print({k: len(v) for k, v in banks.items()})

2026-08-15T17:19:56.021053Z [info     ] sound_bank                     bank_mode=canonical bank_variety=0.5 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:56.227451Z [info     ] sound_bank                     bank_mode=jitter bank_variety=0.5 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:57.107250Z [info     ] sound_bank                     bank_mode=residual bank_variety=0.3 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:57.701704Z [info     ] sound_bank                     bank_mode=stopvar bank_variety=0.5 n=8 seed=3407 steps=50 temperature=1.0


{'canonical': 8, 'jitter_0.5': 8, 'residual_0.3': 8, 'stopvar': 8}


## Measure the gate metrics per mode

Pairwise waveform L1 (diversity), centroid spread (Hz), RMS
(degeneracy guard) — the same metrics as the pre-registered gate
(FAD-proxy omitted here for speed; see `scripts/bank_variants.py`
for the full protocol).

In [4]:
def bank_metrics(clips):
    l1 = [
        float((clips[a] - clips[b]).abs().mean())
        for a in range(len(clips))
        for b in range(a + 1, len(clips))
    ]
    cents = [float(spectral_centroid(c).mean()) for c in clips]
    rms = float(
        torch.cat([c.pow(2).mean().unsqueeze(0) for c in clips]).sqrt().mean()
    )
    return {
        'L1_mean': round(statistics.mean(l1), 4),
        'L1_min': round(min(l1), 4),
        'L1_max': round(max(l1), 4),
        'spread_hz': round(statistics.stdev(cents), 1),
        'centroid_hz': round(statistics.mean(cents), 1),
        'RMS': round(rms, 3),
    }

rows = []
for name, clips in banks.items():
    m = bank_metrics([c.cpu() for c in clips])
    m.update(mode=name, gate='USEFUL' if (
        m['L1_mean'] >= 0.05 and m['RMS'] >= 0.35 and m['spread_hz'] > 20
    ) else '-')
    rows.append(m)

print(f"{'mode':<12} {'L1':>7} {'spread':>7} {'RMS':>6}  gate")
for r in rows:
    print(f"{r['mode']:<12} {r['L1_mean']:>7} {r['spread_hz']:>7} "
          f"{r['RMS']:>6}  {r['gate']}")

mode              L1  spread    RMS  gate
canonical        0.0     0.0  0.546  -
jitter_0.5     0.182    41.2  0.452  USEFUL
residual_0.3   0.112   191.8  0.467  USEFUL
stopvar        0.073    51.1  0.541  USEFUL


Expected (from `results/bank_variants.csv`, gate-passing arms):

- `canonical`: L1 ≈ 0.000 — the contraction; every draw the same clip
- `jitter_0.5`: L1 ≈ 0.182, spread ≈ 41 Hz
- `residual_0.3`: L1 ≈ 0.112, spread ≈ 192 Hz
- `stopvar`: L1 ≈ 0.073, spread ≈ 51 Hz

## Sweep the `bank_variety` dial (jitter)

`bank_variety` is the 0–1 diversity slider (LSD-studio mapping:
dropdown = `bank_mode`, slider = `bank_variety`). Diversity should be
monotone in the dial, with 0 collapsing to the canonical clip.

In [5]:
sweep = []
for a in (0.0, 0.05, 0.1, 0.25, 0.5):
    bank = model.generate_sound_bank(
        n=N_BANK, steps=STEPS, seed=SEED,
        bank_mode='jitter', bank_variety=a,
    )
    m = bank_metrics([c.cpu() for c in bank])
    m.update(alpha=a)
    sweep.append(m)

print(f"{'alpha':>6} {'L1':>7} {'spread':>7} {'RMS':>6}")
for r in sweep:
    print(f"{r['alpha']:>6} {r['L1_mean']:>7} {r['spread_hz']:>7} {r['RMS']:>6}")

2026-08-15T17:19:57.982197Z [info     ] sound_bank                     bank_mode=jitter bank_variety=0.0 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:58.188210Z [info     ] sound_bank                     bank_mode=jitter bank_variety=0.05 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:58.372109Z [info     ] sound_bank                     bank_mode=jitter bank_variety=0.1 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:58.552535Z [info     ] sound_bank                     bank_mode=jitter bank_variety=0.25 n=8 seed=3407 steps=50 temperature=1.0


2026-08-15T17:19:58.703990Z [info     ] sound_bank                     bank_mode=jitter bank_variety=0.5 n=8 seed=3407 steps=50 temperature=1.0


 alpha      L1  spread    RMS
   0.0     0.0     0.0  0.546
  0.05   0.052     4.8  0.539
   0.1  0.0836     9.3  0.514
  0.25  0.1292    18.6  0.474
   0.5   0.182    41.2  0.452


## Audition: write WAVs

One subdirectory per mode, `NN.wav` per clip. Listen and judge the
qualitative claim behind the gate: a coherent palette (same frozen
manifold) with audible intra-bank variation — vs the canonical bank's
n identical clips.

In [6]:
for name, clips in banks.items():
    mode_dir = OUT_DIR / name
    mode_dir.mkdir(parents=True, exist_ok=True)
    for i, clip in enumerate(clips):
        sf.write(str(mode_dir / f'{i:02d}.wav'), clip.squeeze(0).cpu().numpy(), 24000)
print(f'wrote banks under {OUT_DIR}')
for p in sorted(OUT_DIR.glob('*/*.wav'))[:4]:
    print(' ', p.relative_to(REPO))

wrote banks under /Users/tuned-silicon/papers/latent-sound-diffusion/notebooks/results/bank_variants
  notebooks/results/bank_variants/canonical/00.wav
  notebooks/results/bank_variants/canonical/01.wav
  notebooks/results/bank_variants/canonical/02.wav
  notebooks/results/bank_variants/canonical/03.wav


## Cross-check against the experiment record

`results/bank_variants.csv` is the frozen decision record (13 arms).
The wired library modes must reproduce its gate-arm rows bit-for-bit.

In [7]:
import csv

csv_path = REPO / 'results' / 'bank_variants.csv'
with open(csv_path) as f:
    record = {r['arm']: r for r in csv.DictReader(f)}

for name, arm in (
    ('jitter_0.5', 'jitter_a0.5'),
    ('residual_0.3', 'resid_r0.3'),
    ('stopvar', 'stopvar'),
):
    rec = record[arm]
    row = next(r for r in rows if r['mode'] == name)
    match = abs(row['L1_mean'] - float(rec['l1_mean'])) < 0.005
    print(f"{name:<12} notebook L1 {row['L1_mean']:.4f}  "
          f"csv {float(rec['l1_mean']):.4f}  match={match}")

jitter_0.5   notebook L1 0.1820  csv 0.1820  match=True
residual_0.3 notebook L1 0.1120  csv 0.1120  match=True
stopvar      notebook L1 0.0730  csv 0.0727  match=True


## Reproducibility & interface notes

- Same seed → identical banks per mode (unit-tested in
  `tests/test_inference.py::TestBankModes`).
- `bank_variety=0` reduces `jitter`/`residual` exactly to the canonical
  clip — the slider's left end is a safe "same as Stock" anchor.
- Invalid `bank_mode` raises `ValueError`; the UI should bind its
  dropdown directly to the exported `BANK_MODES` tuple.
- `Bank.from_generation(model, bank_mode=..., bank_variety=...)`
  wraps generation with provenance for studio storage.

Related: [issue #53](https://github.com/tuned-org-uk/latent-sound-diffusion/issues/53) (decision record), [issue #58](https://github.com/tuned-org-uk/latent-sound-diffusion/issues/58) (output-noise follow-up), PR #55 (implementation).